# Landmarks com MediaPipe

O **MediaPipe**, do Google, distribui detectores de landmarks prontos, treinados com muito mais imagens do que conseguiríamos anotar, que rodam em tempo real até em CPU: 21 pontos por mão, 33 no corpo e 478 no rosto.

Com um bom detector de landmarks, muitos problemas ficam fáceis. Para reconhecer um gesto, em vez de uma CNN olhando pixels, que mudam com o fundo e a iluminação, um classificador pequeno olha os 21 pontos da mão. Neste laboratório, vemos um exemplo completo, classificando pedra, papel e tesoura pelos pontos da mão, e depois cada um monta um classificador com o seu próprio dataset.

In [ ]:
import base64
import os
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Os detectores do MediaPipe

Instale com `pip install -U mediapipe`. Cada detector é um arquivo `.task`, baixado uma vez, e recebe uma `mp.Image`. A `load_image` aplica a rotação guardada nos metadados das fotos de celular, para os pontos saírem no lugar certo.

In [ ]:
MODELS_DIR = "data/mediapipe"
MODEL_URLS = {
    "hand": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
    "pose": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task",
    "face": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task",
}


def model_path(name):
    path = os.path.join(MODELS_DIR, f"{name}.task")
    if not os.path.exists(path):
        os.makedirs(MODELS_DIR, exist_ok=True)
        urllib.request.urlretrieve(MODEL_URLS[name], path)
    return path


def load_image(source):
    if source.startswith("http"):
        path = os.path.join("data", os.path.basename(source))
        if not os.path.exists(path):
            urllib.request.urlretrieve(source, path)
        source = path
    return ImageOps.exif_transpose(Image.open(source)).convert("RGB")


def to_mp_image(image):
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=np.asarray(image))

São três detectores. O de **mãos** devolve 21 pontos por mão: o punho (0) e quatro pontos por dedo, da base até a ponta. O de **pose** devolve 33 pontos do corpo, cada um com uma `visibility`, a confiança de que o ponto aparece na foto. O de **rosto** devolve 478 pontos e, com `output_face_blendshapes=True`, 52 escores de expressão, como sorrir ou piscar.

Cada ponto tem $x$ e $y$ **normalizados** entre 0 e 1, divididos pela largura e pela altura da imagem, e um $z$, uma estimativa da profundidade.

In [ ]:
def base_options(name):
    return mp_python.BaseOptions(model_asset_path=model_path(name))


hand_detector = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(base_options=base_options("hand"), num_hands=2))
pose_detector = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(base_options=base_options("pose")))
face_detector = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(base_options=base_options("face"), output_face_blendshapes=True)
)

hands_image = load_image("https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg")
person_image = load_image("https://storage.googleapis.com/mediapipe-assets/business-person.png")
hands = hand_detector.detect(to_mp_image(hands_image))
pose = pose_detector.detect(to_mp_image(person_image))
face = face_detector.detect(to_mp_image(person_image))

print(f"{len(hands.hand_landmarks)} mãos, {len(pose.pose_landmarks[0])} pontos de pose, {len(face.face_landmarks[0])} pontos de rosto")
print(hands.hand_landmarks[0][0])   # o punho da primeira mão

A `to_pixels` converte os pontos de uma detecção em um array em pixels, e a `draw_landmarks` desenha os pontos ligados pelas conexões que o MediaPipe define para cada detector, como os ossos de cada dedo.

In [ ]:
HAND_CONNECTIONS = vision.HandLandmarksConnections.HAND_CONNECTIONS
POSE_CONNECTIONS = vision.PoseLandmarksConnections.POSE_LANDMARKS
FACE_CONNECTIONS = vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS


def to_pixels(landmarks, image):
    W, H = image.size
    return np.array([[point.x * W, point.y * H] for point in landmarks])   # [K, 2]


def draw_landmarks(ax, points, connections, point_size=12):
    for connection in connections:
        ax.plot(points[[connection.start, connection.end], 0], points[[connection.start, connection.end], 1], color="lime", linewidth=1)
    ax.scatter(points[:, 0], points[:, 1], s=point_size, c="red", zorder=3)


fig, axes = plt.subplots(1, 3, figsize=(15, 7))
for ax, image in zip(axes, [hands_image, person_image, person_image]):
    ax.imshow(image)
    ax.axis("off")

for landmarks in hands.hand_landmarks:
    draw_landmarks(axes[0], to_pixels(landmarks, hands_image), HAND_CONNECTIONS)
axes[0].set_title("mãos: 21 pontos por mão")

draw_landmarks(axes[1], to_pixels(pose.pose_landmarks[0], person_image), POSE_CONNECTIONS)
axes[1].set_title("pose: 33 pontos")

face_points = to_pixels(face.face_landmarks[0], person_image)
draw_landmarks(axes[2], face_points, FACE_CONNECTIONS, point_size=1)
axes[2].set_xlim(face_points[:, 0].min() - 40, face_points[:, 0].max() + 40)   # aproxima o rosto
axes[2].set_ylim(face_points[:, 1].max() + 40, face_points[:, 1].min() - 40)
axes[2].set_title("rosto: 478 pontos")
plt.show()

A foto mostra só a metade de cima do corpo, mas a pose estima os 33 pontos mesmo assim, e as pernas são desenhadas abaixo da imagem. É a `visibility` que diz em quais pontos confiar. No rosto, os blendshapes de maior escore descrevem a expressão.

In [ ]:
for k, name in [(0, "nariz"), (11, "ombro esquerdo"), (25, "joelho esquerdo"), (27, "tornozelo esquerdo")]:
    print(f"{name}: visibility {pose.pose_landmarks[0][k].visibility:.2f}")

for category in sorted(face.face_blendshapes[0], key=lambda category: category.score, reverse=True)[:3]:
    print(f"{category.category_name}: {category.score:.2f}")

## Exemplo: pedra, papel e tesoura

O dataset **Rock Paper Scissors**, do TensorFlow, tem 2520 imagens de treino e 372 de teste, uma pasta por classe. As mãos foram geradas por computação gráfica, e as do teste vêm de modelos diferentes das do treino.

In [ ]:
rps_dir = "data/rps"
for name in ["rps", "rps-test-set"]:
    if not os.path.exists(os.path.join(rps_dir, name)):
        os.makedirs(rps_dir, exist_ok=True)
        zip_path = os.path.join(rps_dir, f"{name}.zip")
        urllib.request.urlretrieve(f"https://storage.googleapis.com/download.tensorflow.org/data/{name}.zip", zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(rps_dir)

rps_classes = sorted(os.listdir(os.path.join(rps_dir, "rps")))

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, label in zip(axes, rps_classes):
    folder = os.path.join(rps_dir, "rps", label)
    ax.imshow(load_image(os.path.join(folder, sorted(os.listdir(folder))[0])))
    ax.set_title(label)
    ax.axis("off")
plt.show()

### Extraindo os pontos

O dataset do classificador são os pontos, e não as imagens. A `extract_hand` devolve os 21 pontos da primeira mão encontrada, em pixels, para a forma da mão não sair distorcida em fotos que não são quadradas.

Usamos só o $x$ e o $y$. O $z$ é uma estimativa de profundidade feita a partir de uma única foto, pouco confiável. Neste dataset, um classificador com o $z$ acerta 100% na validação e só 77% no teste, porque o $z$ das mãos de teste é bem diferente do das mãos de treino.

A extração leva cerca de um minuto.

In [ ]:
def extract_hand(image):
    result = hand_detector.detect(to_mp_image(image))
    if not result.hand_landmarks:
        return None
    return to_pixels(result.hand_landmarks[0], image)   # [21, 2] em pixels


def extract_dataset(root):
    X, y = [], []
    for label_index, label in enumerate(rps_classes):
        folder = os.path.join(root, label)
        for name in sorted(os.listdir(folder)):
            points = extract_hand(load_image(os.path.join(folder, name)))
            if points is not None:   # sem mão detectada, a imagem fica de fora
                X.append(points)
                y.append(label_index)
    return np.stack(X), np.array(y)


X_train, y_train = extract_dataset(os.path.join(rps_dir, "rps"))
X_test, y_test = extract_dataset(os.path.join(rps_dir, "rps-test-set"))
print(X_train.shape, X_test.shape)

### Normalização

As coordenadas em pixels dependem de onde a mão está e de quão grande ela aparece, e nada disso tem a ver com o gesto. Subtraímos o punho de todos os pontos e dividimos pela maior distância de um ponto ao punho:

$$
\tilde{p}_k = \frac{p_k - p_0}{\max_j \left\| p_j - p_0 \right\|}
$$

Os 21 pontos normalizados viram um vetor de 42 valores, a entrada do classificador.

In [ ]:
def normalize_hands(X):
    X = X - X[:, :1]                                        # punho na origem
    scale = np.linalg.norm(X, axis=2).max(axis=1)           # maior distância ao punho, [N]
    return (X / scale[:, None, None]).reshape(len(X), -1)   # [N, 42]


features_train = torch.tensor(normalize_hands(X_train), dtype=torch.float32)
features_test = torch.tensor(normalize_hands(X_test), dtype=torch.float32)
print(features_train.shape)

### O classificador

Uma rede com duas camadas de 64 neurônios basta. Separamos 15% do treino para a validação e deixamos o teste, com mãos de outros modelos, para o fim.

In [ ]:
generator = torch.Generator().manual_seed(0)
order = torch.randperm(len(features_train), generator=generator)
n_val = int(0.15 * len(order))
val_idx, train_idx = order[:n_val], order[n_val:]

labels_train = torch.tensor(y_train)
train_dataloader = DataLoader(TensorDataset(features_train[train_idx], labels_train[train_idx]), batch_size=64, shuffle=True)
val_dataloader = DataLoader(TensorDataset(features_train[val_idx], labels_train[val_idx]), batch_size=256)
test_dataloader = DataLoader(TensorDataset(features_test, torch.tensor(y_test)), batch_size=256)


def accuracy(model, dataloader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in dataloader:
            correct += (model(x.to(device)).argmax(dim=1).cpu() == y).sum().item()
    return correct / len(dataloader.dataset)

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(42, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, len(rps_classes)),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
num_epochs = 50
losses, val_accuracies = [], []

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for x, y in train_dataloader:
        loss = criterion(model(x.to(device)), y.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    losses.append(total_loss / len(train_dataloader))
    val_accuracies.append(accuracy(model, val_dataloader))

print(f"acurácia de validação {val_accuracies[-1]:.3f}")

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
ax_loss.plot(range(1, num_epochs + 1), losses)
ax_loss.set_title("perda de treino")
ax_loss.set_xlabel("época")
ax_acc.plot(range(1, num_epochs + 1), val_accuracies)
ax_acc.set_title("acurácia de validação")
ax_acc.set_xlabel("época")
plt.show()

print(f"acurácia de teste {accuracy(model, test_dataloader):.3f}")

### Testando com fotos reais

O classificador foi treinado com mãos geradas por computador, mas ele só vê os pontos. Se o detector encontra os pontos de uma mão real, o classificador funciona do mesmo jeito. A `classify_hand` recebe o caminho ou a URL de uma foto, desenha os pontos e mostra a probabilidade de cada classe.

In [ ]:
def classify_hand(source):
    image = load_image(source)
    points = extract_hand(image)
    if points is None:
        print("nenhuma mão detectada")
        return

    x = torch.tensor(normalize_hands(points[None]), dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        probabilities = model(x)[0].softmax(dim=0).cpu()

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(image)
    draw_landmarks(ax, points, HAND_CONNECTIONS)
    ax.set_title(", ".join(f"{label} {p:.2f}" for label, p in zip(rps_classes, probabilities)))
    ax.axis("off")
    plt.show()


classify_hand("https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg")

### Usando a webcam

A `take_photo` tira uma foto com a webcam e salva em um arquivo. No Colab, o notebook roda em um servidor sem câmera, então a foto é tirada pelo navegador, com um trecho de JavaScript que mostra a imagem da câmera e um botão; o Python recebe a foto codificada em base64. Rodando na sua máquina, a foto vem direto da câmera pelo OpenCV.

Rode a célula, faça um dos gestos na frente da câmera e clique em **Capturar**.

In [ ]:
TAKE_PHOTO_JS = '''
async function takePhoto() {
  const div = document.createElement("div");
  const button = document.createElement("button");
  button.textContent = "Capturar";
  const video = document.createElement("video");
  video.style.display = "block";
  div.appendChild(video);
  div.appendChild(button);
  document.body.appendChild(div);

  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  video.srcObject = stream;
  await video.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  await new Promise((resolve) => button.onclick = resolve);

  const canvas = document.createElement("canvas");
  canvas.width = video.videoWidth;
  canvas.height = video.videoHeight;
  canvas.getContext("2d").drawImage(video, 0, 0);
  stream.getVideoTracks()[0].stop();
  div.remove();
  return canvas.toDataURL("image/jpeg", 0.9);
}
'''


def take_photo(path="data/webcam.jpg"):
    try:
        from google.colab.output import eval_js
    except ImportError:   # fora do Colab, lê a câmera direto
        import cv2

        camera = cv2.VideoCapture(0)
        ok, frame = camera.read()
        camera.release()
        if not ok:
            raise RuntimeError("não foi possível ler a webcam")
        cv2.imwrite(path, frame)
        return path

    from IPython.display import Javascript, display

    display(Javascript(TAKE_PHOTO_JS))
    data = eval_js("takePhoto()")   # "data:image/jpeg;base64,..."
    with open(path, "wb") as f:
        f.write(base64.b64decode(data.split(",")[1]))
    return path

In [ ]:
classify_hand(take_photo())

## O seu classificador

Agora cada um monta um classificador sobre landmarks para um problema seu, do começo ao fim. O código é por sua conta; as funções do exemplo podem ser reaproveitadas e adaptadas.

### Exercício: o problema

Escolha um problema com 3 a 6 classes que dependa da forma da mão, do corpo ou do rosto. Algumas ideias: letras estáticas de Libras, números com os dedos, posturas de yoga, postura sentada correta ou curvada, expressões faciais. Evite classes que dependem de movimento, como acenar: uma foto não registra movimento.

*Descreva aqui o problema e as classes.*

### Exercício: as fotos e os pontos

Fotografe de 40 a 60 imagens por classe, em `data/meu_classificador/<classe>/`. Varie a distância, o ângulo, o fundo e a iluminação, e, se possível, fotografe mais de uma pessoa. Extraia os pontos com o detector adequado e confira, desenhando os pontos sobre algumas fotos. Em quantas fotos o detector falhou?

In [ ]:
# extraia os pontos de todas as fotos e desenhe alguns exemplos de cada classe

### Exercício: a normalização

Adapte a normalização ao seu problema. Qual ponto serve de origem e qual distância serve de escala? No corpo, o centro dos quadris e a largura dos ombros são escolhas comuns. Vale a pena girar os pontos para eliminar a inclinação, ou a inclinação faz parte da classe, como em um joinha para cima e para baixo?

In [ ]:
# escreva a normalização e monte a entrada do classificador

### Exercício: treino e avaliação

Separe treino, validação e teste. Fotos tiradas em sequência são quase iguais, e se uma cai no treino e a vizinha no teste, o teste fica fácil demais. Se fotografou mais de uma pessoa, deixe uma pessoa inteira para o teste. Treine, mostre a acurácia de teste e algumas fotos que o modelo errou. Teste também com fotos novas e com uma foto fora das classes: o que o modelo responde?

In [ ]:
# separe os dados, treine e avalie no teste

*Comente aqui os resultados: quais classes o modelo confunde, e os erros vêm do classificador ou do detector?*